# 01 DQA Core Probe 1h

This notebook is a small, non-federated-loop probe for the core DQA-CWA idea.

The full `03` notebook tests DQA inside the full FedSTO-style loop. This notebook asks a narrower question: if we take one current global checkpoint, train each environment client once on a small subset, and aggregate those client updates with DQA-CWA instead of plain FedAvg, does the class-wise aggregation itself look helpful?

It writes everything under `dynamic_quality_aware_classwise_aggregation/exploring/runs/dqa_core_probe_1h/` and does not modify the main `03` workspace.

## Safety Notes

This is intentionally not a replacement for the full experiment. It is a quick plausibility probe.

- It does not run multiple federated rounds.
- It uses small sampled lists, so the absolute mAP is noisy.
- The useful signal is the comparison between `source_global`, `fedavg_probe`, and `dqa_probe` under the same tiny protocol.
- If the full `03` run is active, training is skipped by default to avoid stealing GPU memory.

In [ ]:
from __future__ import annotations

import importlib
import json
import os
import re
import shlex
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Optional

import pandas as pd


def find_repo_root(start: Optional[Path] = None) -> Path:
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    required = (
        'dynamic_quality_aware_classwise_aggregation/run_dqa_cwa_fedsto.py',
        'navigating_data_heterogeneity/setup_fedsto_exact_reproduction.py',
    )
    candidate_dirs = []
    for base in (start, *start.parents):
        candidate_dirs.extend([base, base / 'Object_Detection', base / 'masters_research' / 'Object_Detection'])
    for candidate in candidate_dirs:
        if all((candidate / marker).exists() for marker in required):
            return candidate.resolve()
    raise FileNotFoundError('Could not locate Object_Detection repository root.')


REPO_ROOT = find_repo_root()
DQA_ROOT = REPO_ROOT / 'dynamic_quality_aware_classwise_aggregation'
NAV_ROOT = REPO_ROOT / 'navigating_data_heterogeneity'
SOURCE_WORK_ROOT = DQA_ROOT / 'efficientteacher_dqa_cwa_corrected_12h'
SOURCE_STATS_ROOT = DQA_ROOT / 'stats_corrected_12h'
PROBE_ROOT = DQA_ROOT / 'exploring' / 'runs' / 'dqa_core_probe_1h'
PROBE_LIST_ROOT = PROBE_ROOT / 'data_lists'
PROBE_CONFIG_ROOT = PROBE_ROOT / 'configs'
PROBE_GLOBAL_ROOT = PROBE_ROOT / 'global_checkpoints'
PROBE_STATS_ROOT = PROBE_ROOT / 'stats'
PROBE_LOG = PROBE_ROOT / 'dqa_core_probe.log'
EVAL_SCRIPT = DQA_ROOT / 'evaluate_paper_protocol.py'

preferred_python = Path('/root/micromamba/envs/al_yolov8/bin/python')
PYTHON_BIN = preferred_python if preferred_python.exists() else Path(sys.executable)

print('repo_root:', REPO_ROOT)
print('source_work:', SOURCE_WORK_ROOT)
print('probe_root:', PROBE_ROOT)
print('python:', PYTHON_BIN)

## 1. Probe Configuration

The defaults are designed to fit into roughly an hour when the GPUs are free. If the full DQA run is still active, leave `ALLOW_CONCURRENT_FULL_RUN = False` and run this later.

In [ ]:
SAMPLE_CLIENT_IMAGES = 768
SAMPLE_SERVER_TRAIN_IMAGES = 768
SAMPLE_SERVER_VAL_IMAGES = 384
BATCH_SIZE = 64
WORKERS = 0
REQUESTED_GPUS = 2
MASTER_PORT = 29523

RUN_CLIENT_TRAINING = True
RUN_AGGREGATION = True
RUN_SERVER_CALIBRATION = True
RUN_EVALUATION = True
ALLOW_CONCURRENT_FULL_RUN = False

# Keep the quick evaluation small by default. Use cloudy,overcast,rainy,snowy,total for a fuller check.
EVAL_SPLITS = 'cloudy,total'
EVAL_BATCH_SIZE = 16

try:
    import torch

    AVAILABLE_CUDA_GPUS = torch.cuda.device_count()
except Exception as exc:
    AVAILABLE_CUDA_GPUS = 0
    print('Could not inspect CUDA devices:', exc)

GPUS = min(REQUESTED_GPUS, AVAILABLE_CUDA_GPUS) if AVAILABLE_CUDA_GPUS else 1

active_full_run = subprocess.run(
    ['bash', '-lc', "pgrep -af 'run_dqa_cwa_fedsto|torch.distributed.run|train.py' || true"],
    capture_output=True,
    text=True,
    cwd=REPO_ROOT,
).stdout.strip()
active_lines = [
    line
    for line in active_full_run.splitlines()
    if 'dqa_core_probe_1h' not in line
    and 'pgrep -af' not in line
    and 'bash -c pgrep' not in line
]
FULL_RUN_ACTIVE = bool(active_lines)

if FULL_RUN_ACTIVE and not ALLOW_CONCURRENT_FULL_RUN:
    print('Full training appears active; training probe will be skipped unless ALLOW_CONCURRENT_FULL_RUN=True.')
    RUN_CLIENT_TRAINING = False
    RUN_SERVER_CALIBRATION = False

config_summary = {
    'sample_client_images': SAMPLE_CLIENT_IMAGES,
    'sample_server_train_images': SAMPLE_SERVER_TRAIN_IMAGES,
    'sample_server_val_images': SAMPLE_SERVER_VAL_IMAGES,
    'batch_size': BATCH_SIZE,
    'requested_gpus': REQUESTED_GPUS,
    'available_cuda_gpus': AVAILABLE_CUDA_GPUS,
    'gpus_used': GPUS,
    'full_run_active': FULL_RUN_ACTIVE,
    'run_client_training': RUN_CLIENT_TRAINING,
    'run_server_calibration': RUN_SERVER_CALIBRATION,
    'eval_splits': EVAL_SPLITS,
}
config_summary

## 2. Select Source Checkpoint And Build Mini Lists

By default, the source checkpoint is the latest completed global from `03`. The mini lists are sampled from the same weather/domain split, but written into the exploration workspace.

In [ ]:
def read_lines(path: Path) -> list[str]:
    return [line.strip() for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]


def write_first(src: Path, dst: Path, n: int) -> Path:
    lines = read_lines(src)[:n]
    if not lines:
        raise RuntimeError(f'No lines found in {src}')
    dst.parent.mkdir(parents=True, exist_ok=True)
    dst.write_text('\n'.join(lines) + '\n', encoding='utf-8')
    return dst


history_path = SOURCE_WORK_ROOT / 'history.json'
if history_path.exists():
    history = json.loads(history_path.read_text(encoding='utf-8'))
else:
    history = []

if history:
    source_global = Path(history[-1]['global'])
    source_label = f"source_phase{int(history[-1]['phase'])}_round{int(history[-1]['round']):03d}"
else:
    source_global = SOURCE_WORK_ROOT / 'global_checkpoints' / 'round000_warmup.pt'
    source_label = 'source_warmup'

if not source_global.exists():
    raise FileNotFoundError(f'Source checkpoint not found: {source_global}')

PROBE_ROOT.mkdir(parents=True, exist_ok=True)
PROBE_LIST_ROOT.mkdir(parents=True, exist_ok=True)
PROBE_CONFIG_ROOT.mkdir(parents=True, exist_ok=True)
PROBE_GLOBAL_ROOT.mkdir(parents=True, exist_ok=True)
PROBE_STATS_ROOT.mkdir(parents=True, exist_ok=True)

source_lists = SOURCE_WORK_ROOT / 'data_lists'
mini_server_train = write_first(source_lists / 'server_cloudy_train.txt', PROBE_LIST_ROOT / 'probe_server_cloudy_train.txt', SAMPLE_SERVER_TRAIN_IMAGES)
mini_server_val = write_first(source_lists / 'server_cloudy_val.txt', PROBE_LIST_ROOT / 'probe_server_cloudy_val.txt', SAMPLE_SERVER_VAL_IMAGES)

clients = [
    {'id': 0, 'weather': 'overcast'},
    {'id': 1, 'weather': 'rainy'},
    {'id': 2, 'weather': 'snowy'},
]
for client in clients:
    src = source_lists / f"client_{client['id']}_{client['weather']}_target.txt"
    dst = PROBE_LIST_ROOT / f"probe_client_{client['id']}_{client['weather']}_target.txt"
    client['target'] = write_first(src, dst, SAMPLE_CLIENT_IMAGES)

manifest = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'source_global': str(source_global),
    'source_label': source_label,
    'mini_server_train': str(mini_server_train),
    'mini_server_val': str(mini_server_val),
    'clients': [{**client, 'target': str(client['target'])} for client in clients],
}
(PROBE_ROOT / 'probe_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
manifest


## 3. Prepare EfficientTeacher Helpers

The probe reuses the same EfficientTeacher config builder and checkpoint utilities as the full DQA/FedSTO code, but redirects all generated files into the exploration workspace.

In [ ]:
if str(NAV_ROOT) not in sys.path:
    sys.path.insert(0, str(NAV_ROOT))
if str(DQA_ROOT) not in sys.path:
    sys.path.insert(0, str(DQA_ROOT))

setup = importlib.import_module('setup_fedsto_exact_reproduction')
setup.WORK_ROOT = PROBE_ROOT
setup.LIST_ROOT = PROBE_LIST_ROOT
setup.CONFIG_ROOT = PROBE_CONFIG_ROOT
setup.RUN_ROOT = PROBE_ROOT / 'runs'

fedsto = importlib.import_module('run_fedsto_efficientteacher_exact')
fedsto.PRETRAINED_PATH = PROBE_ROOT / 'weights' / 'efficient-yolov5l.pt'
fedsto.GLOBAL_DIR = PROBE_GLOBAL_ROOT
fedsto.CLIENT_STATE_DIR = PROBE_ROOT / 'client_states'
fedsto.HISTORY_PATH = PROBE_ROOT / 'history.json'

from dqa_cwa_aggregation import AggregationConfig, aggregate_checkpoints, load_round_stats


def write_probe_config(name: str, *, target: Path | None, weights: Path, role: str) -> Path:
    cfg = setup.efficientteacher_config(
        name=name,
        train=mini_server_train,
        val=mini_server_val,
        target=target,
        weights=str(weights.resolve()),
        epochs=1,
        train_scope='all',
        orthogonal_weight=0.0,
        batch_size=BATCH_SIZE,
        workers=WORKERS,
        device='' if GPUS > 1 else '',
    )
    cfg['project'] = str((PROBE_ROOT / 'runs').resolve())
    if role == 'server':
        cfg['SSOD'] = {'train_domain': False}
    return setup.write_config(f'{name}.yaml', cfg)


def run_train(config: Path, *, extra_env: dict[str, str] | None = None) -> Path:
    if GPUS > 1:
        cmd = [
            str(PYTHON_BIN), '-m', 'torch.distributed.run', '--nproc_per_node', str(GPUS),
            '--master_port', str(MASTER_PORT), 'train.py', '--cfg', str(config.resolve())
        ]
    else:
        cmd = [str(PYTHON_BIN), 'train.py', '--cfg', str(config.resolve())]

    print(' '.join(shlex.quote(part) for part in cmd))
    env = os.environ.copy()
    if extra_env:
        env.update(extra_env)
    PROBE_LOG.parent.mkdir(parents=True, exist_ok=True)
    with PROBE_LOG.open('a', encoding='utf-8') as log:
        log.write('\n' + '=' * 100 + '\n')
        log.write(' '.join(cmd) + '\n')
        subprocess.run(cmd, cwd=fedsto.setup.ET_ROOT, env=env, stdout=log, stderr=subprocess.STDOUT, check=True)

    import yaml
    with config.open(encoding='utf-8') as f:
        run_name = yaml.safe_load(f)['name']
    return fedsto.checkpoint_path(run_name)


print('probe log:', PROBE_LOG)


## 4. One Small Client-Update Pass

This is the only extra training part. It trains each client once from the same source global checkpoint on a small unlabeled subset and exports the pseudo-label class stats that DQA uses.

In [ ]:
probe_source_global = PROBE_GLOBAL_ROOT / f'{source_label}_global.pt'
if not probe_source_global.exists():
    fedsto.make_start_checkpoint(source_global, probe_source_global, protocol='dqa_core_probe_v1', stage='source_global')

client_checkpoints = []
if RUN_CLIENT_TRAINING:
    for client in clients:
        run_name = f"probe_client{client['id']}_{client['weather']}"
        start = PROBE_GLOBAL_ROOT / f"{run_name}_start.pt"
        if not start.exists():
            fedsto.make_start_checkpoint(probe_source_global, start, protocol='dqa_core_probe_v1', stage=f'{run_name}_start')

        stats_out = PROBE_STATS_ROOT / f"probe_client{client['id']}.json"
        cfg = write_probe_config(run_name, target=client['target'], weights=start, role='client')
        ckpt = run_train(
            cfg,
            extra_env={
                'DQA_PSEUDO_STATS_OUT': str(stats_out.resolve()),
                'DQA_CLIENT_ID': str(client['id']),
                'DQA_PHASE': 'probe',
                'DQA_ROUND': '1',
            },
        )
        fedsto.mark_checkpoint_protocol(ckpt, 'dqa_core_probe_v1', f'{run_name}_client_update')
        client['checkpoint'] = ckpt
        client['stats'] = stats_out
        client_checkpoints.append(ckpt)
else:
    for client in clients:
        ckpt = fedsto.checkpoint_path(f"probe_client{client['id']}_{client['weather']}")
        stats_out = PROBE_STATS_ROOT / f"probe_client{client['id']}.json"
        if ckpt.exists() and stats_out.exists():
            client['checkpoint'] = ckpt
            client['stats'] = stats_out
            client_checkpoints.append(ckpt)

if len(client_checkpoints) != len(clients):
    print('Client probe checkpoints are not complete yet. Run this cell when GPUs are free, or set RUN_CLIENT_TRAINING=True.')
else:
    print('client_checkpoints:')
    for path in client_checkpoints:
        print(' ', path)

## 5. Aggregate FedAvg vs DQA-CWA

This is the core comparison. Both variants use the same client checkpoints. `fedavg_probe` is ordinary averaging. `dqa_probe` changes only classification-head rows according to pseudo-label count/confidence reliability.

In [ ]:
aggregate_outputs = {}
round_stats = PROBE_STATS_ROOT / 'probe_round001.json'

if RUN_AGGREGATION and len(client_checkpoints) == len(clients):
    payload = {'clients': []}
    for client in clients:
        item = json.loads(Path(client['stats']).read_text(encoding='utf-8'))
        if isinstance(item, dict) and 'clients' in item:
            payload['clients'].extend(item['clients'])
        else:
            payload['clients'].append(item)
    round_stats.write_text(json.dumps(payload, indent=2), encoding='utf-8')

    fedavg_out = PROBE_GLOBAL_ROOT / 'fedavg_probe.pt'
    dqa_out = PROBE_GLOBAL_ROOT / 'dqa_probe.pt'
    dqa_anchor_out = PROBE_GLOBAL_ROOT / 'dqa_probe_server_anchor_1p0.pt'

    fedsto.aggregate_checkpoints(client_checkpoints, probe_source_global, fedavg_out, backbone_only=False)
    stats = load_round_stats(round_stats, num_classes=10)
    aggregate_checkpoints(
        client_checkpoints=client_checkpoints,
        server_checkpoint=probe_source_global,
        output_checkpoint=dqa_out,
        stats=stats,
        state_path=PROBE_ROOT / 'dqa_probe_state.json',
        config=AggregationConfig(num_classes=10),
        repo_root=REPO_ROOT,
    )
    aggregate_checkpoints(
        client_checkpoints=client_checkpoints,
        server_checkpoint=probe_source_global,
        output_checkpoint=dqa_anchor_out,
        stats=stats,
        state_path=PROBE_ROOT / 'dqa_probe_anchor_state.json',
        config=AggregationConfig(num_classes=10, server_anchor=1.0),
        repo_root=REPO_ROOT,
    )
    aggregate_outputs = {
        'source_global': probe_source_global,
        'fedavg_probe': fedavg_out,
        'dqa_probe': dqa_out,
        'dqa_probe_anchor1': dqa_anchor_out,
    }

if aggregate_outputs:
    for label, path in aggregate_outputs.items():
        print(label, path, path.exists())
else:
    print('No aggregate outputs yet.')

## 6. Optional One-Epoch Server Calibration

This adds the same labeled-server correction to each aggregate once, without entering a federated loop. It is useful because the real FedSTO/DQA protocol always lets the server repair the aggregate on labeled cloudy data.

In [ ]:
calibrated_outputs = dict(aggregate_outputs)

if RUN_SERVER_CALIBRATION and aggregate_outputs:
    for label, checkpoint in aggregate_outputs.items():
        if label == 'source_global':
            continue
        run_name = f'probe_server_calib_{label}'
        cfg = write_probe_config(run_name, target=None, weights=checkpoint, role='server')
        ckpt = run_train(cfg)
        fedsto.mark_checkpoint_protocol(ckpt, 'dqa_core_probe_v1', f'{run_name}_server_calibration')
        out = PROBE_GLOBAL_ROOT / f'{label}_server_calibrated.pt'
        fedsto.make_start_checkpoint(ckpt, out, protocol='dqa_core_probe_v1', stage=f'{label}_server_calibrated')
        calibrated_outputs[f'{label}_server_calibrated'] = out

if calibrated_outputs:
    for label, path in calibrated_outputs.items():
        print(label, path, path.exists())
else:
    print('No calibrated outputs yet.')

## 7. Quick Paper-Style Evaluation

The default evaluates only `cloudy,total` to keep the probe short. If DQA looks better than FedAvg here, that is useful evidence for the full run. If it is worse, inspect the pseudo-label stats before trusting the full loop.

In [ ]:
checkpoints = calibrated_outputs or aggregate_outputs
eval_cmd = [
    str(PYTHON_BIN), str(EVAL_SCRIPT),
    '--workspace', str(PROBE_ROOT),
    '--splits', EVAL_SPLITS,
    '--batch-size', str(EVAL_BATCH_SIZE),
]
for label, path in checkpoints.items():
    if Path(path).exists():
        eval_cmd.extend(['--checkpoint', f'{label}={Path(path).resolve()}'])

if RUN_EVALUATION and checkpoints:
    print('Running:', ' '.join(shlex.quote(part) for part in eval_cmd))
    subprocess.run(eval_cmd, cwd=REPO_ROOT, check=True)
else:
    print('Evaluation skipped. Set RUN_EVALUATION=True after aggregate checkpoints exist.')

summary_path = PROBE_ROOT / 'validation_reports' / 'paper_protocol_eval_summary.csv'
if summary_path.exists():
    eval_summary = pd.read_csv(summary_path)
    display(eval_summary)
else:
    print('No evaluation summary yet:', summary_path)

## 8. Read The Signal

A quick rule of thumb:

- If `dqa_probe` beats `fedavg_probe` on `total` or on non-cloudy splits, the DQA core is doing something useful.
- If `dqa_probe_server_calibrated` beats `fedavg_probe_server_calibrated`, that is the most relevant signal for the full protocol.
- If DQA loses badly, inspect `probe_round001.json`: the pseudo-label reliability may be rewarding noisy classes or overconfident false positives.

In [ ]:
summary_path = PROBE_ROOT / 'validation_reports' / 'paper_protocol_eval_summary.csv'
if summary_path.exists():
    df = pd.read_csv(summary_path)
    ok = df[df['status'] == 'ok'].copy()
    if ok.empty:
        display(df)
    else:
        display(ok.pivot_table(index='checkpoint_label', columns='split', values='map50', aggfunc='first').round(4))
        display(ok.pivot_table(index='checkpoint_label', columns='split', values='map50_95', aggfunc='first').round(4))

        total = ok[ok['split'] == 'total'].copy()
        if not total.empty:
            display(total[['checkpoint_label', 'precision', 'recall', 'map50', 'map50_95']].round(4))
else:
    print('Run the evaluation cell first.')

if round_stats.exists():
    stats_payload = json.loads(round_stats.read_text(encoding='utf-8'))
    print('stats file:', round_stats)
    print(json.dumps(stats_payload, indent=2)[:4000])

## Artifacts

The main outputs are below.

In [ ]:
artifact_rows = []
for label, path in {
    'probe_manifest': PROBE_ROOT / 'probe_manifest.json',
    'probe_log': PROBE_LOG,
    'round_stats': round_stats if 'round_stats' in globals() else PROBE_STATS_ROOT / 'probe_round001.json',
    'eval_summary': PROBE_ROOT / 'validation_reports' / 'paper_protocol_eval_summary.csv',
    'global_checkpoints': PROBE_GLOBAL_ROOT,
    'runs': PROBE_ROOT / 'runs',
}.items():
    path = Path(path)
    artifact_rows.append({
        'label': label,
        'path': str(path),
        'exists': path.exists(),
        'size_mib': round((sum(p.stat().st_size for p in path.rglob('*') if p.is_file()) if path.is_dir() else path.stat().st_size if path.exists() else 0) / 1024**2, 2),
    })
display(pd.DataFrame(artifact_rows))